# Data Analysis

Explore historical price data, compute indicators, and inspect signal behaviour for a single ticker.

**Sections**
1. Setup & data fetch
2. Price & volume overview
3. Technical indicators deep-dive
4. Signal analysis
5. Portfolio snapshot

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from src.automated_trading_system import AutomatedTradingSystem
from src.indicator_calculator import IndicatorCalculator

TICKER     = 'AAPL'
START_DATE = '2023-01-01'
END_DATE   = '2024-01-01'

print('Libraries loaded.')

## 1. Fetch Data & Run Backtest

In [ ]:
system = AutomatedTradingSystem(
    initial_capital=10_000,
    ticker=TICKER,
    max_positions=5,
    max_position_size_pct=0.05,
)
results = system.backtest(start_date=START_DATE, end_date=END_DATE)

df = system.indicators['daily'].copy()
print(f'Loaded {len(df)} daily bars  ({df.index[0].date()} → {df.index[-1].date()})')
print(f'Columns: {list(df.columns)}')

## 2. Price & Volume Overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
fig.suptitle(f'{TICKER} — Price & Volume', fontsize=13, fontweight='bold')

ax1, ax2 = axes

ax1.fill_between(df.index, df['BB_Lower'], df['BB_Upper'], alpha=0.15, color='steelblue', label='Bollinger Bands')
ax1.plot(df.index, df['Close'],  color='#1f77b4', linewidth=1.5, label='Close')
ax1.plot(df.index, df['SMA_20'], color='orange',  linewidth=1.0, label='SMA 20')
ax1.plot(df.index, df['SMA_50'], color='green',   linewidth=1.0, label='SMA 50')
ax1.plot(df.index, df['SMA_200'],color='red',     linewidth=1.0, linestyle='--', label='SMA 200')
ax1.set_ylabel('Price ($)')
ax1.legend(fontsize=8, ncol=3)
ax1.grid(alpha=0.3)

ax2.bar(df.index, df['Volume'], color='#aec7e8', width=0.8)
ax2.set_ylabel('Volume')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../runs/nb_price_volume.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved to runs/nb_price_volume.png')

## 3. Technical Indicators Deep-Dive

In [ ]:
# Summary statistics for each indicator
indicator_cols = ['RSI', 'MACD', 'MACD_Histogram', 'ADX', 'ATR', 'Stoch_K', 'Stoch_D']
stats = df[indicator_cols].describe().T[['mean', 'std', 'min', 'max']].round(3)
print('Indicator Statistics:')
print(stats.to_string())

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f'{TICKER} — Technical Indicators', fontsize=13, fontweight='bold')

# RSI
ax = axes[0]
ax.plot(df.index, df['RSI'], color='purple', linewidth=1.2)
ax.axhline(70, color='red',   linewidth=0.8, linestyle='--', alpha=0.7)
ax.axhline(30, color='green', linewidth=0.8, linestyle='--', alpha=0.7)
ax.fill_between(df.index, 70, df['RSI'].clip(lower=70), color='red',   alpha=0.15)
ax.fill_between(df.index, df['RSI'].clip(upper=30), 30, color='green', alpha=0.15)
ax.set_ylabel('RSI'); ax.set_ylim(0, 100); ax.grid(alpha=0.3)
ax.set_title('RSI (14)', fontsize=9)

# MACD
ax = axes[1]
hist = df['MACD_Histogram']
ax.bar(df.index, hist.where(hist >= 0), color='green', alpha=0.7, width=0.8)
ax.bar(df.index, hist.where(hist <  0), color='red',   alpha=0.7, width=0.8)
ax.plot(df.index, df['MACD'],        color='blue',   linewidth=1.0, label='MACD')
ax.plot(df.index, df['MACD_Signal'], color='orange', linewidth=1.0, label='Signal')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('MACD'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax.set_title('MACD (12/26/9)', fontsize=9)

# ADX
ax = axes[2]
ax.plot(df.index, df['ADX'],      color='purple', linewidth=1.2, label='ADX')
ax.plot(df.index, df['Plus_DI'],  color='green',  linewidth=0.9, linestyle='--', label='+DI')
ax.plot(df.index, df['Minus_DI'], color='red',    linewidth=0.9, linestyle='--', label='-DI')
ax.axhline(25, color='grey', linewidth=0.7, linestyle=':')
ax.set_ylabel('ADX'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax.set_title('ADX / Directional Index (14)', fontsize=9)

# Stochastic
ax = axes[3]
ax.plot(df.index, df['Stoch_K'], color='blue',   linewidth=1.0, label='%K')
ax.plot(df.index, df['Stoch_D'], color='orange', linewidth=1.0, linestyle='--', label='%D')
ax.axhline(80, color='red',   linewidth=0.7, linestyle='--', alpha=0.7)
ax.axhline(20, color='green', linewidth=0.7, linestyle='--', alpha=0.7)
ax.set_ylim(0, 100); ax.set_ylabel('Stoch'); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax.set_title('Stochastic (14)', fontsize=9)

plt.tight_layout()
plt.savefig('../runs/nb_indicators.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Signal Analysis

In [ ]:
signals = system.signals_history
sig_summary = system.signal_generator.get_signal_summary()

print(f'Total signals : {sig_summary["total_signals"]}')
print(f'Buy signals   : {sig_summary["buy_signals"]}')
print(f'Sell signals  : {sig_summary["sell_signals"]}')
print(f'Avg confidence: {sig_summary["avg_confidence"]:.1%}')

if signals:
    sig_df = pd.DataFrame([
        {'date': s.timestamp, 'type': s.signal_type, 'confidence': s.confidence,
         'entry': s.entry_price, 'stop': s.stop_loss, 'target': s.take_profit}
        for s in signals
    ]).set_index('date')
    print(f'\nLast 5 signals:')
    print(sig_df.tail(5).to_string())

In [ ]:
if signals:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'{TICKER} — Signal Distribution', fontsize=12, fontweight='bold')

    # Confidence histogram
    ax = axes[0]
    buys  = [s.confidence for s in signals if s.signal_type == 'BUY']
    sells = [s.confidence for s in signals if s.signal_type == 'SELL']
    ax.hist(buys,  bins=10, color='green', alpha=0.7, label=f'Buy ({len(buys)})')
    ax.hist(sells, bins=10, color='red',   alpha=0.7, label=f'Sell ({len(sells)})')
    ax.axvline(0.55, color='black', linewidth=1, linestyle='--', label='Threshold')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
    ax.set_title('Signal Confidence Distribution'); ax.legend(fontsize=8)

    # Buy vs Sell pie
    ax = axes[1]
    ax.pie([len(buys), len(sells)], labels=[f'Buy\n{len(buys)}', f'Sell\n{len(sells)}'],
           colors=['green', 'red'], autopct='%1.0f%%', startangle=90)
    ax.set_title('Buy / Sell Split')

    plt.tight_layout()
    plt.savefig('../runs/nb_signal_dist.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No signals generated for this period.')

## 5. Portfolio Snapshot

In [ ]:
summary  = system.portfolio.get_portfolio_summary()
tr_stats = system.trade_logger.get_trade_statistics()

print('=== Portfolio Summary ===')
for k, v in summary.items():
    print(f'  {k:25s}: {v}')

print('\n=== Trade Statistics ===')
for k, v in tr_stats.items():
    print(f'  {k:25s}: {v}')

In [ ]:
closed = system.portfolio.closed_positions
if closed:
    trade_df = pd.DataFrame([
        {'entry_date': t.entry_date, 'exit_date': t.exit_date, 'side': t.side,
         'entry_price': t.entry_price, 'exit_price': t.exit_price,
         'quantity': t.quantity, 'pnl': t.profit_loss, 'return_pct': t.return_pct,
         'holding_days': t.holding_days, 'win': t.win}
        for t in closed
    ])
    print(f'Closed trades: {len(trade_df)}')
    print(trade_df[['side','entry_price','exit_price','pnl','return_pct','holding_days','win']].to_string())
else:
    print('No closed trades.')